# Suite2p Pipeline for Two-Photon Calcium Imaging (suite2p 1.0: `settings` + `db`)

This notebook runs the full [suite2p](https://github.com/MouseLand/suite2p) pipeline with the **1.0 API**, in which the old flat `ops` dictionary is split in two:

| | `db` | `settings` |
|--|--|--|
| **Describes** | the *dataset*: where the data are, how many planes / channels / ROIs, which files | *how to process it*: registration, detection, extraction, … |
| **Shape** | flat dict | nested dict (`settings["registration"]`, `settings["detection"]`, …) |
| **Defaults** | `default_db()` | `default_settings()` |
| **Reuse** | one per recording | one preset shared by many recordings |

`run_s2p(db=db, settings=settings)` runs everything, and writes `db.npy` + `settings.npy` into the results folder. At the end this notebook also saves the settings as a JSON preset, which the batch and mesoscope notebooks can load.

The notebook first defines `db` (section 2), then `settings` (section 3), one cell per functional group. Parameters carried over from the old `ops` notebook keep their explanations and memory/VRAM notes; the old keys that no longer exist are collected in 3.12. A full old → new key table is in the appendix.

**Reference:** Pachitariu et al. 2017 | Docs: https://suite2p.readthedocs.io | Code: https://github.com/MouseLand/suite2p

In [ ]:
# %pip install suite2p

import suite2p
from suite2p import run_s2p, default_settings, default_db
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json

print(f"suite2p version: {suite2p.__version__}")

import copy

def deep_update(base, new):
    """Recursively merge `new` into `base` (nested dicts are merged, not replaced).
    run_s2p() only does a shallow merge, so a partial group such as
    {"registration": {"nonrigid": False}} would otherwise wipe the other
    registration defaults."""
    for k, v in new.items():
        if isinstance(v, dict) and isinstance(base.get(k), dict):
            deep_update(base[k], v)
        else:
            base[k] = v
    return base

def find_unknown_keys(d, ref, prefix=""):
    """Keys in `d` that do not exist in the reference dict (typos / removed keys)."""
    bad = []
    for k, v in d.items():
        if k not in ref:
            bad.append(prefix + k)
        elif isinstance(v, dict) and isinstance(ref[k], dict):
            bad += find_unknown_keys(v, ref[k], prefix + k + ".")
    return bad

def to_jsonable(v):
    if isinstance(v, dict):            return {k: to_jsonable(x) for k, x in v.items()}
    if isinstance(v, (list, tuple)):   return [to_jsonable(x) for x in v]
    if isinstance(v, np.ndarray):      return v.tolist()
    if isinstance(v, np.generic):      return v.item()
    if isinstance(v, Path):            return str(v)
    return v

def load_settings_file(path):
    """Load a settings file: a .json written by these notebooks, or a settings.npy
    written by suite2p into any suite2p/ or suite2p/planeN/ folder."""
    path = Path(path)
    assert path.exists(), f"Settings file not found: {path}"
    if path.suffix == ".json":
        with open(path) as f:
            return json.load(f)
    if path.suffix == ".npy":
        return np.load(path, allow_pickle=True).item()
    raise ValueError(f"Unsupported format: {path.suffix}")

## 1. Data paths

In [ ]:
DATA_PATH = [r"/path/to/your/raw/data"]
SAVE_PATH = r"/path/to/your/processed"
FAST_DISK = r""   # fast SSD scratch; "" = default (same location as the results)
SETTINGS_NAME = "my_settings"   # the settings JSON preset is saved as <SAVE_PATH>/<SETTINGS_NAME>.json

for p in DATA_PATH:
    assert Path(p).exists(), f"Data path not found: {p}"
Path(SAVE_PATH).mkdir(parents=True, exist_ok=True)
print("Paths OK")

## 2. Database (`db`)

Describes the **dataset only**: where the data are, how many planes / channels / ROIs they contain. No processing parameters here.

In [ ]:
db = default_db()

### 2.1 Files, paths and binary handling

In [ ]:
db["data_path"] = DATA_PATH
db["save_path0"] = SAVE_PATH   # suite2p creates a 'suite2p' subdirectory here
db["fast_disk"] = FAST_DISK or None

# Name of the results subdirectory inside save_path0.
# Default: "suite2p".
db["save_folder"] = "suite2p"

# Whether to also search one level of subdirectories inside each data_path.
# Default: False.
db["look_one_level_down"] = False

# Optional: restrict the search to these named child folders only.
# None = search all subfolders.  Default: None.
db["subfolders"] = None

# Whether to use scikit-image (sklearn.external.tifffile) for TIFF reading
# instead of the default ScanImage TIFF reader.  Try True if the default
# reader fails to open your files.
# ⚠️ HIDDEN — not in official docs.  Default: False.
db["force_sktiff"] = False

# Bruker multiplane ordering: True = bidirectional (0,1,2,2,1,0),
# False = sequential (0,1,2,0,1,2).  Only relevant when bruker=True.
# ⚠️ HIDDEN — not in official docs.  Default: False.
db["bruker_bidirectional"] = False

# Dataset key inside the HDF5 file that holds the imaging array.
db["h5py_key"] = "data"

# HDF5 driver for the NWB file (empty string = local file, no special driver).
# ⚠️ HIDDEN — not in official docs.  Default: "".
db["nwb_driver"] = ""

# TwoPhotonSeries name inside the NWB file.  Defaults to the first series found.
# ⚠️ HIDDEN — not in official docs.  Default: "".
db["nwb_series"] = ""

# Whether to keep the unregistered raw binary on disk.  Needed to view raw vs.
# registered movies in the GUI and required for two_step_registration.
# Default: False.  (The official docs say True — that is WRONG; source is False.)
#
# 💾 DISK IMPACT — creates a second full copy of the binary (data_raw.bin).
#   Extra disk per plane (20 min @ 30 Hz = 36 000 frames):
#     512×512  :  +36 GB   (data.bin = 36 GB → total = 72 GB)
#     1000×2000: +288 GB   (data.bin = 288 GB → total = 576 GB)
#   Only enable if you need the GUI raw/registered comparison or two_step_registration.
db["keep_movie_raw"] = False

# To exclude frames during ROI detection (e.g. photostimulation artefacts),
# save a NumPy integer array called 'bad_frames.npy' in db['data_path'][0].

### 2.2 Acquisition geometry (planes, channels, ROIs)

In [ ]:
# Number of imaging planes interleaved in the data files.  1 = single-plane.
# Default: 1.
#
# 💾 DATA TYPE — ScanImage saves raw TIFFs as int16 (2 bytes/pixel).
#   Suite2p converts every frame to float32 (4 bytes/pixel) when writing
#   data.bin.  This doubles the per-frame size relative to the TIFF files
#   sitting on disk:
#
#     Format              512×512 frame    1000×2000 frame
#     ────────────────── ─────────────── ─────────────────
#     Raw TIFF  (int16)      0.5 MB/frame      4 MB/frame
#     data.bin  (float32)    1.0 MB/frame      8 MB/frame
#
#   All disk and RAM estimates throughout this notebook use the float32 sizes.
#
# 💾 DISK IMPACT — multiplies all binary files by nplanes.
#   Each plane gets its own data.bin.  RAM impact is per-plane (processed
#   one at a time), so nplanes does NOT multiply peak RAM.
#   data.bin size per plane (20 min @ 30 Hz = 36 000 frames):
#     512×512  :  36 000 × 1 MB =  36 GB / plane
#     1000×2000:  36 000 × 8 MB = 288 GB / plane
#   nplanes = 4 on 1000×2000 → ~1.15 TB total on disk.  Ensure you have space.
db["nplanes"] = 1

# Number of channels per plane.  Data must be channel-interleaved:
# (t0,ch1), (t0,ch2), (t1,ch1), (t1,ch2), …
# Default: 1.
#
# 💾 DISK IMPACT — doubles all binary files when nchannels=2.
#   Both channels are written to data.bin; during registration only the
#   functional channel is loaded, so batch_size RAM is unchanged.
#   Extra disk per plane (20 min @ 30 Hz):
#     512×512  :  +36 GB  (now 72 GB total)
#     1000×2000: +288 GB  (now 576 GB total)
db["nchannels"] = 1

# Which channel (1-based) contains the functional calcium signal.
# Default: 1.
db["functional_chan"] = 1

# List of planes (0-based) to skip, e.g. flyback planes in a volumetric acquisition.
# None = process all planes.  Default: None.
db["ignore_flyback"] = None

# Leave at the defaults for ordinary single-FoV recordings.  For MROI data these
# come from the MATLAB helper mesoscope_json_from_scanimage.m — see the
# mesoscope notebook.  All four live in db (they describe the data).
db["nrois"] = 1      # number of ScanImage ROIs per file

db["lines"] = None   # per ROI: row indices of the raw frame belonging to it
db["dx"]    = None   # per ROI: x pixel offset in the stitched FoV
db["dy"]    = None   # per ROI: y pixel offset in the stitched FoV

## 3. Settings (`settings`)

Describes **how to process** the data, split into the nested groups of `default_settings()`. No dataset information here.

In [ ]:
settings = default_settings()

### 3.1 General  (`settings[...]`)

In [ ]:
# Device used for registration / detection.  Default: "cuda".
# Set to "cpu" to avoid GPU-memory limits (slower; see the VRAM notes in the registration cell).
settings["torch_device"] = "cuda"

# Volume rate PER PLANE in Hz.  For multi-plane recordings divide the volume
# rate by the number of planes.  Example: 30-Hz volume, 3 planes → fs = 10.
# Default: 10.0.  ← This is used for deconvolution kernel scaling.
settings["fs"] = 30.0   # ← set to your actual per-plane frame rate

# Expected cell diameter in pixels, [y, x].  Used by Cellpose; different from
# detection["sparsery_settings"]["spatial_scale"].
# Default in 1.0: [12.0, 12.0].
settings["diameter"] = [12.0, 12.0]

# GCaMP decay time constant (seconds). OASIS uses this to build its kernel;
# the kernel is NOT fitted to the data, so setting this correctly matters.
# Recommended values:
#   GCaMP6f  → 0.7 s
#   GCaMP6m  → 1.0 s  (default)
#   GCaMP6s  → 1.25–1.5 s
#   jGCaMP7f → ~0.5 s
#   jGCaMP8  → 0.2–0.4 s
# Default: 1.0.
settings["tau"] = 1.0

### 3.2 `run` — which stages to run

In [ ]:
# Whether to run planes in parallel on a compute cluster / multi-GPU server.
# ⚠️ HIDDEN — experimental.  Default: False.
settings["run"]["multiplane_parallel"] = False

# Whether to run registration.  Default: True.
settings["run"]["do_registration"] = 1

# Whether to run ROI detection.  Default: True.
settings["run"]["do_detection"] = True

# Whether to run spike deconvolution.  Default: True.
settings["run"]["do_deconvolution"] = True

### 3.3 `io` — output options

In [ ]:
# Whether to delete the intermediate binary after the full pipeline finishes.
# Keep False while tuning parameters so you can rerun without re-converting TIFFs.
# Default: False.
settings["io"]["delete_bin"] = False

# When fast_disk differs from save_path0, move (rather than copy) the binary
# from fast_disk to save_path0 when processing is complete.
# ⚠️ HIDDEN — not in official docs.  Default: False.
settings["io"]["move_bin"] = False

# Save results as a MATLAB Fall.mat file in addition to .npy files.
# Default: False.
settings["io"]["save_mat"] = False

# Export results to NWB format.  Default: False.
settings["io"]["save_NWB"] = False

# After processing each plane separately, merge all planes into a single
# 'suite2p/combined' folder so the GUI can display them together.
# Default: True.
settings["io"]["combined"] = True

### 3.4 `registration` — motion correction

In [ ]:
# Which channel (1-based) to use for motion registration.
# 1 = functional (default); 2 = anatomical.
# Using a stable structural channel can improve registration quality.
# Default: 1.
settings["registration"]["align_by_chan2"] = False   # was align_by_chan (1 = functional, 2 = chan 2)

# Whether to estimate and correct the phase offset between odd and even scan
# lines (bidirectional resonant scanning artefact).  suite2p estimates the
# offset from the first nimg_init frames and applies it to all subsequent frames.
# Default: False.
settings["registration"]["do_bidiphase"] = False

# Fixed bidirectional phase offset in pixels.  Any non-zero value is applied
# directly, overriding do_bidiphase estimation.
# Default: 0.
settings["registration"]["bidiphase"] = 0

# Number of initial frames used to build the reference image.  These frames are
# first quickly pre-aligned, then averaged.
# Default in 1.0: 400.  (The old flat ops default was 300.)
#
# 💾 MEMORY IMPACT: LOW — safe to increase freely on either frame size.
#   ScanImage saves as int16 (2 bytes/pixel) → float32 (4 bytes/pixel) internally.
#   Frame sizes:                int16 (raw TIFF)    float32 (internal)
#     512×512                     0.5 MB/frame         1 MB/frame
#     1000×2000                   4.0 MB/frame         8 MB/frame
#
#   nimg_init frames loaded at once:
#     512×512,   nimg_init=300 : 300 ×  1 MB =  0.3 GB  (negligible)
#     1000×2000, nimg_init=300 : 300 ×  8 MB =  2.4 GB  (~3% of 96 GB — still fine)
#
#   GUIDANCE: use 300–1000.  More frames → better reference → better registration.
#   Diminishing returns beyond ~500.  Memory is NOT the limiting factor here.
settings["registration"]["nimg_init"] = 400

# Number of frames processed per GPU/CPU registration batch.
# Default in 1.0: 100.  (The old flat ops default was 500.  Note that 1.0 has
# separate batch sizes: settings["extraction"]["batch_size"] = 500 for signal
# extraction, and db["batch_size"] = 500 for the TIFF -> binary conversion.)
#
# ⚠️ GPU VRAM IS THE REAL BOTTLENECK — not system RAM.
#   When a CUDA GPU is available, suite2p runs registration on the GPU.
#   Each batch must fit in VRAM.  The raw-frame + FFT buffers alone account for
#   ~2× frame_size per frame, but non-rigid registration (nonrigid=True, default)
#   adds per-block phase-correlation maps that dominate on large frames.
#
#   With nonrigid=True and block_size=[128,128], a 1000×2000 frame splits into
#   ~128 blocks.  Suite2p computes a correlation map per block per frame, plus
#   reference copies and PyTorch's allocator overhead.  The actual VRAM footprint
#   can be 10–30× higher than the naive "frames × frame_size" estimate, and it
#   scales with both batch_size and frame area.
#
#   Empirical guidance for a 12 GB GPU with nonrigid=True (block_size=[128,128]):
#
#     512×512 frames:                    1000×2000 frames:
#       batch_size =  500 →  safe ✓        batch_size = 500 → CRASH ✗
#       batch_size = 1000 →  safe ✓        batch_size = 100 → may crash ✗
#       batch_size = 1500 →  borderline ⚠️  batch_size =  50 → borderline ⚠️
#       batch_size = 2000 →  CRASH ✗       batch_size =  30 → safe ✓ (empirical)
#
#   GUIDANCE on a 12 GB GPU:
#     512×512  : keep default (500).  Can push to ~1200–1500.
#     1000×2000: start at 30 and increase until stable.  The optimal value
#                is dataset-dependent — nonrigid block count and frame content
#                both matter.  Start low and binary-search upward.
#
#   Two levers to relax the VRAM constraint WITHOUT reducing batch_size:
#     1. nonrigid=False  — eliminates block maps entirely; only 2–3 full-frame
#                          buffers needed → allows much larger batch_size.
#     2. block_size=[256,256] — cuts block count from ~128 to ~32 for 1000×2000
#                          → significantly less VRAM per frame.
#
#   NOTE: system RAM (96 GB) is NOT the constraint here.  VRAM is.
#   If you force CPU-only (ops["gpu_port"] = 0 or no CUDA), batch_size can be
#   hundreds–thousands for either frame size without crashing system RAM.
settings["registration"]["batch_size"] = 100

# Maximum allowed rigid shift as a fraction of the frame dimension.
# The pixel limit applied is: max(Ly, Lx) * maxregshift.
# Default: 0.1 (10% of frame size).
settings["registration"]["maxregshift"] = 0.1

# Gaussian smoothing sigma (pixels) applied to both the frame and the reference
# before computing the phase-correlation.  For 1P data at 512×512 the docs
# recommend > 4.  Default: 1.15.
settings["registration"]["smooth_sigma"] = 1.15

# Gaussian smoothing sigma applied across the temporal dimension (frames) before
# phase-correlation.  Helps when individual frames have very low SNR.
# 0 = disabled.  Try 1–2 for noisy datasets.  Default: 0.
settings["registration"]["smooth_sigma_time"] = 0

# Whether to normalize frames (zero-mean, unit-variance) before computing
# the phase-correlation.  Reduces bias from bright static structures.
# ⚠️ HIDDEN — not in official docs.  Default: True.
settings["registration"]["norm_frames"] = True

# Threshold that determines which frames are considered 'bad' (high-motion)
# and excluded when computing the final crop region of the registered movie.
# LOWER values exclude MORE frames.  Stored as ops['badframes'] after registration.
# ⚠️ HIDDEN — not in official docs.  Default: 1.0.
settings["registration"]["th_badframes"] = 1.0

# Subpixel resolution for the registration: 1/subpixel pixels.
# 10 = register to 0.1 pixel accuracy.  Default: 10.
settings["registration"]["subpixel"] = 10

# Run registration twice: first pass uses the original binary, second pass
# uses the output of the first to build a better reference image.
# Useful for low-SNR data.  Requires keep_movie_raw = True.  Default: False.
settings["registration"]["two_step_registration"] = False

# Whether to perform non-rigid (block-wise) registration in addition to global
# rigid.  Non-rigid corrects local tissue deformations (cardiac pulse, z-drift).
# Default: True.
#
# ⚠️ GPU VRAM: setting this to False dramatically reduces VRAM usage during
#   registration.  With nonrigid=False, only 2–3 full-frame buffers are needed
#   per batch.  For 1000×2000 frames on a 12 GB GPU, consider nonrigid=False
#   if you cannot tolerate a very small batch_size, then inspect quality.
settings["registration"]["nonrigid"] = True

# Block size [y, x] in pixels for non-rigid registration.  Powers of 2 (128,
# 256, 384) are recommended for efficient FFTs.  Smaller blocks → finer
# correction but slower and more noise-sensitive.  Default: [128, 128].
#
# ⚠️ GPU VRAM: smaller blocks = more blocks = more per-block FFT maps = more VRAM.
#   For 1000×2000: [128,128] → ~128 blocks; [256,256] → ~32 blocks (~4× less VRAM).
#   Try [256,256] if batch_size=30 is too slow and you need a larger batch.
settings["registration"]["block_size"] = [128, 128]

# SNR threshold for accepting a block's shift estimate (peak-to-noise ratio of
# the phase-correlation map).  Blocks below this are interpolated from neighbors.
# For 1P data ~1.5.  Default: 1.2.
settings["registration"]["snr_thresh"] = 1.2

# Maximum allowed block shift in pixels, relative to the global rigid shift.
# Prevents blocks from drifting far from the whole-frame solution.  Default: 5.
settings["registration"]["maxregshiftNR"] = 5

# Pixels at each edge set to zero (tapered) before registration, to reduce
# vignetting artefacts.  Must be ≥ 3 × smooth_sigma.  Default: 40.
settings["registration"]["spatial_taper"] = 40

# Write the registered functional channel to TIFF files on disk (large!).
# Default: False.
#
# 💾 DISK IMPACT — writes the full registered movie as uint16 TIFF (2 bytes/px).
#   20 min @ 30 Hz = 36 000 frames:
#     512×512  :  36 000 × 0.5 MB =  18 GB
#     1000×2000:  36 000 × 4.0 MB = 144 GB
settings["registration"]["reg_tif"] = False

# Same for the second (anatomical) channel.  Default: False.
# 💾 Same disk impact as reg_tif above.
settings["registration"]["reg_tif_chan2"] = False

### 3.5 `detection` — ROI detection (common)

In [ ]:
# Use the sparse-mode (sparsery) cell detection algorithm.
# True (default) = sparsery, designed for sparsely active neurons.
# False = sourcery, the original dense algorithm (slower).
# NOTE: The official docs say default is False — that is WRONG; source is True.
settings["detection"]["algorithm"] = "sparsery"   # was sparse_mode: True = "sparsery", False = "sourcery"

# Maximum number of binned frames used to compute the covariance matrix that
# drives ROI detection.  suite2p bins every B = n_frames / nbinned frames into
# one, then loads the ENTIRE binned movie into RAM at once for correlation-based
# cell finding.  This is the most dangerous parameter for memory.
# Default: 5000.
#
# ⚠️  MEMORY IMPACT: HIGH — this is THE parameter most likely to crash your machine.
#
#   After registration, suite2p stores data.bin as float32 (4 bytes/pixel).
#   The binned movie is held ENTIRELY in system RAM during detection (CPU step):
#     RAM = nbinned × Ly × Lx × 4 bytes
#
#   For 512×512 frames (1 MB/frame float32):
#     nbinned =  5 000 →   5 GB  ← DEFAULT  (fine on 96 GB, <6% of RAM)
#     nbinned = 10 000 →  10 GB             (fine on 96 GB)
#     nbinned = 20 000 →  20 GB             (fine on 96 GB)
#     nbinned = 40 000 →  40 GB             (fine on 96 GB, ~42%)
#     nbinned = 80 000 →  80 GB             (safe maximum on 96 GB)
#     nbinned = 96 000 →  96 GB  ← CRASH    (no headroom for OS + other processes)
#
#   For 1000×2000 frames (8 MB/frame float32):
#     nbinned =  5 000 →  40 GB  ← DEFAULT  (~42% of 96 GB — manageable)
#     nbinned =  8 000 →  64 GB             (~67% of 96 GB — tight but OK)
#     nbinned = 10 000 →  80 GB             (~83% of 96 GB — risky)
#     nbinned = 11 000 →  88 GB             (very risky — OOM likely with OS overhead)
#     nbinned = 12 000 →  96 GB  ← CRASH
#
#   GUIDANCE on a 96 GB machine:
#     512×512  : keep nbinned ≤ 80 000.  Default (5 000) is very conservative;
#                increasing to 20 000–40 000 can slightly improve detection
#                quality for short or low-SNR recordings.
#     1000×2000: the default (5 000) already uses 40 GB.  Keep it ≤ 8 000.
#                If you also enable denoise=True, halve those limits (see below).
#
#   IMPORTANT: nbinned is the BINNED frame count, not the raw frame count.
#   For a 20-min recording at 30 Hz = 36 000 raw frames with nbinned=5000:
#     binsize ≈ 7 → ~5 143 binned frames → memory stays ~40 GB regardless of
#     recording length.  Longer recordings just bin more frames per bin.
settings["detection"]["nbins"] = 5000

# Multiplier applied to the automatically determined activity threshold when
# searching for cell seeds.
# HIGHER → fewer, more confident ROIs.
# LOWER  → more ROIs, including small/dim cells, but also more false positives.
# Default: 1.0.  (Official docs say 5.0 — that is WRONG; source is 1.0.)
settings["detection"]["threshold_scaling"] = 1.0

# Running-mean window in frames subtracted from each pixel's trace before
# correlation-based cell finding.  Removes slow bleaching/drift trends.
# For 1P data the docs recommend setting below 10.  Default: 100.
settings["detection"]["highpass_time"] = 100

# Maximum fraction of a cell's pixels that may overlap another already-detected
# cell before the new ROI is discarded.  1.0 = accept all.  Default: 0.75.
settings["detection"]["max_overlap"] = 0.75

# Whether to denoise the binned movie with PCA before detection (sparse_mode only).
# Can improve detection in very noisy recordings but adds compute time.
# ⚠️ Not prominently documented.  Default: False.
#
# ⚠️  MEMORY IMPACT: HIGH — roughly DOUBLES the RAM of the detection step.
#   PCA denoising reshapes the binned movie into a 2D matrix and computes
#   the covariance; the working arrays are approximately the same size as
#   the binned movie itself.
#
#   For 512×512 frames (1 MB/frame):
#     nbinned= 5 000 → binned movie  5 GB → PCA adds  ~5 GB → total ~10 GB  ✓
#     nbinned=40 000 → binned movie 40 GB → PCA adds ~40 GB → total ~80 GB  ✓ (96 GB)
#     nbinned=48 000 → binned movie 48 GB → PCA adds ~48 GB → total ~96 GB  ✗ CRASH
#
#   For 1000×2000 frames (8 MB/frame):
#     nbinned= 5 000 → binned movie 40 GB → PCA adds ~40 GB → total ~80 GB  ⚠️ risky
#     nbinned= 3 000 → binned movie 24 GB → PCA adds ~24 GB → total ~48 GB  ✓ safe
#
#   GUIDANCE on a 96 GB machine:
#     512×512  : denoise=True is safe up to nbinned ≈ 40 000.
#     1000×2000: only enable if nbinned ≤ 3 000.  With the default nbinned=5 000
#                the detection step alone uses 40 GB; adding PCA pushes to ~80 GB,
#                leaving very little headroom.  Lower nbinned first, then enable.
settings["detection"]["denoise"] = False

# Crop dendrites for classifier statistics like compactness.  Set True (default)
# to get cleaner soma-only shape statistics; False to include dendrites.
# Default: True.
settings["detection"]["soma_crop"] = True

# Threshold for calling an ROI detected on the second (anatomical) channel.
# suite2p computes the ratio F_chan2 / F_chan1; ROIs above this are red-positive.
# Only relevant when nchannels == 2.  Default: 0.65.
settings["detection"]["chan2_threshold"] = 0.65

### 3.6 `detection["sparsery_settings"]`

In [ ]:
# Expected cell scale in pixels.  Controls the spatial filter used to find seeds.
# 0 = automatic multi-scale (recommended on first attempt).
# 1 ≈ 6 px | 2 ≈ 12 px | 3 ≈ 24 px | 4 ≈ 48 px.
# Default: 0.
settings["detection"]["sparsery_settings"]["spatial_scale"] = 0

# Window (in pixels) for the uniform spatial high-pass filter applied to the
# binned movie BEFORE detection, to subtract local neuropil background.
# This is DIFFERENT from spatial_hp_reg (which is for registration).
# ⚠️ HIDDEN — not in official docs.  Default: 25.
settings["detection"]["sparsery_settings"]["highpass_neuropil"] = 25

### 3.7 `detection["sourcery_settings"]`

In [ ]:
# Only keep pixels that are connected (8-connected) to the brightest pixel of
# each ROI.  Set False to allow fragmented ROIs (useful for dendrites/boutons).
# Default: True.
settings["detection"]["sourcery_settings"]["connected"] = True

# Maximum number of detection iterations (the algorithm stops earlier if no
# more peaks are above threshold).  Default: 20.
settings["detection"]["sourcery_settings"]["max_iterations"] = 20

### 3.8 `detection["cellpose_settings"]`

In [ ]:
# Cellpose cell probability threshold.  Higher = stricter (fewer cells).
# ⚠️ HIDDEN — not in official docs.  Default: 0.0.
settings["detection"]["cellpose_settings"]["cellprob_threshold"] = 0.0

# Cellpose flow error threshold.  Higher = more permissive (more cells accepted).
# ⚠️ HIDDEN — not in official docs.  Default: 0.4.
settings["detection"]["cellpose_settings"]["flow_threshold"] = 0.4

# Apply a spatial high-pass filter to the image before Cellpose, expressed as a
# multiple of the diameter.  0 = no filtering.  Helps in dense neuropil.
# ⚠️ HIDDEN — not in official docs.  Default: 0.
settings["detection"]["cellpose_settings"]["highpass_spatial"] = 0

# Cellpose model to use.  Can be a built-in model name ('cpsam', 'cyto3',
# 'cyto2', 'nuclei', …) or a path to a custom trained model.
# 'cpsam' = Cellpose SAM (Segment Anything), the current default.
# ⚠️ HIDDEN — not in official docs.  Default: 'cpsam'.
settings["detection"]["cellpose_settings"]["cellpose_model"] = "cpsam"

### 3.9 `classification`

In [ ]:
# Apply the cell classifier DURING detection.  ROIs with probability below this
# value are discarded early, before full signal extraction — speeds up the
# pipeline when there are many non-cell ROIs.  Set to 0.0 to disable.
# Default: 0.0.
settings["classification"]["preclassify"] = 0.0

# Use suite2p's built-in logistic-regression classifier (trained on mouse V1
# recordings) to label ROIs as cell / non-cell after extraction.
# NOTE: Default is False in source — many tutorials incorrectly say True.
settings["classification"]["use_builtin_classifier"] = False

# Path to a custom classifier file (.npy) trained via the suite2p GUI.
# Leave as "" to use the built-in model when use_builtin_classifier is True.
settings["classification"]["classifier_path"] = ""

### 3.10 `extraction` — signals and neuropil

In [ ]:
# Whether to compute and save Fneu (neuropil fluorescence).
# False = Fneu is set to zero for all cells.  Default: True.
settings["extraction"]["neuropil_extract"] = True

# Gap between the edge of the cell mask and the inner edge of the neuropil
# sampling annulus, in pixels.  Increase if you see somatic contamination in Fneu.
# Default: 2.
settings["extraction"]["inner_neuropil_radius"] = 2

# Minimum number of pixels in the neuropil annulus.  Cells near the image edge
# that have fewer available pixels use a larger annulus.  Default: 350.
settings["extraction"]["min_neuropil_pixels"] = 350

# Percentile of pixel weights (lam) within the cell area to use as a cutoff:
# pixels below this percentile are EXCLUDED from neuropil estimation to prevent
# dim cell pixels contaminating the neuropil signal.
# ⚠️ HIDDEN — not in official docs.  Default: 50.0.
settings["extraction"]["lam_percentile"] = 50.0

# When two ROIs share pixels, include those pixels in both ROIs' traces.
# False (default) = exclude shared pixels from both.  Default: False.
settings["extraction"]["allow_overlap"] = False

# Neuropil contamination fraction:
#   F_corrected = F_cell − neucoeff × F_neuropil
# Empirically estimated as 0.65–0.75 for GCaMP6 (Kerlin et al. 2010).
# Set to 0.0 to skip neuropil subtraction entirely.  Default: 0.7.
settings["extraction"]["neuropil_coefficient"] = 0.7

### 3.11 `dcnv_preprocess` — baseline for deconvolution

In [ ]:
# Method used to estimate the fluorescence baseline (F0) before deconvolution.
#
# 'maximin' (default): robust rolling baseline.
#   Step 1 – Gaussian-smooth the trace with sigma = sig_baseline × fs.
#   Step 2 – compute the minimum of a rolling window of width win_baseline × fs.
#   Step 3 – compute the maximum of that rolling minimum.
#   This baseline tracks slow drifts while ignoring transient calcium events.
#
# 'constant': Gaussian-smooth with sig_baseline × fs, then take the minimum
#   of the whole smoothed trace.  Simple but doesn't track slow bleaching.
#
# 'constant_prctile': take the prctile_baseline percentile of the whole trace.
#   Fastest but most affected by very active cells.
settings["dcnv_preprocess"]["baseline"] = "maximin"

# Rolling window length in seconds for the maximin baseline computation.
# Should be longer than the longest expected calcium transient.  Default: 60.0 s.
settings["dcnv_preprocess"]["win_baseline"] = 60.0

# Gaussian smoothing sigma (seconds) applied to the trace before baseline
# estimation (used in both 'maximin' and 'constant' methods).  Default: 10.0 s.
settings["dcnv_preprocess"]["sig_baseline"] = 10.0

# Percentile used when baseline = 'constant_prctile'.
# 8th percentile ≈ low, robust estimate of baseline.  Default: 8.0.
settings["dcnv_preprocess"]["prctile_baseline"] = 8.0

### 3.12 Old `ops` keys that no longer exist

In [ ]:
# Reference only — NOT used.  Keys of the old flat `ops` that have no 1.0 equivalent,
# with the explanation each one had in the old notebook.

# Whether to process Bruker Prairie View single-page TIFFs.
# suite2p will split files by channel based on filenames ('Ch1' = channel 1).
# Default: False.
# [1.0] ops["bruker"] has no direct equivalent: Bruker input is now chosen with db["input_format"] (check the accepted value for Bruker in the 1.0 docs).
# ops["bruker"] = False

# Whether to read ScanImage mesoscope files (special multi-ROI format).
# Default: False.
# [1.0] ops["mesoscan"] has no direct equivalent: there is no mesoscan flag in db; multi-ROI data is described by db["nrois"], db["lines"], db["dx"], db["dy"] (see the mesoscope notebook).
# ops["mesoscan"] = False

# List of HDF5 file paths. Leave as [] if file discovery should handle them.
# [1.0] ops["h5py"] has no direct equivalent: HDF5 input is now chosen with db["input_format"] (check the accepted value for HDF5 in the 1.0 docs) plus db["h5py_key"].
# ops["h5py"] = []

# Path to a NWB file (deactivates data_path discovery).
# ⚠️ HIDDEN — not in official docs.  Default: "".
# [1.0] ops["nwb_file"] has no direct equivalent: NWB input is now chosen with db["input_format"] (check the accepted value for NWB in the 1.0 docs).
# ops["nwb_file"] = ""

# If > 0, process only this many frames per plane.  Useful for quick tests.
# Default: -1 (all frames).
# [1.0] ops["frames_include"] has no direct equivalent: not present in the 1.0 default_db() / default_settings().
# ops["frames_include"] = -1

# Whether bidirectional correction has already been applied to the data
# (e.g. by the acquisition software).  When True, suite2p skips correction.
# ⚠️ HIDDEN — not in official docs.  Default: False.
# [1.0] ops["bidi_corrected"] has no direct equivalent: not present in the 1.0 default_db() / default_settings().
# ops["bidi_corrected"] = False

# Pixel aspect ratio: µm per pixel in X / µm per pixel in Y.
# Used only by the GUI for correct display aspect ratio.  Default: 1.0.
# [1.0] ops["aspect"] has no direct equivalent: GUI-only display key; not present in the 1.0 defaults.
# ops["aspect"] = 1.0

# If True, and a reference image (ops['refImg']) is already stored in an ops
# file, skip computing a new one and use the stored image directly.
# ⚠️ HIDDEN — not in official docs.  Default: False.
# [1.0] ops["force_refImg"] has no direct equivalent: not present in the 1.0 default_settings().
# ops["force_refImg"] = False

# Whether to pad the frame with zeros during the FFT shift step.
# Can reduce wrap-around artefacts at the frame edges.
# ⚠️ HIDDEN — not in official docs.  Default: False.
# [1.0] ops["pad_fft"] has no direct equivalent: not present in the 1.0 default_settings().
# ops["pad_fft"] = False

# Enable high-pass spatial filtering and edge tapering before each registration
# step.  Removes slow background gradients that corrupt phase-correlation.
# Set True for 1P data; leave False for standard 2P.  Default: False.
# [1.0] ops["1Preg"] has no direct equivalent: not present in the 1.0 default_settings() registration group.
# ops["1Preg"] = False

# Window size in pixels for the spatial high-pass filter used in 1P mode
# (subtracts a boxcar-average of this width before computing shifts).
# Default: 42.  NOTE: in the source this key is 'spatial_hp_reg', not 'spatial_hp'.
# [1.0] ops["spatial_hp_reg"] has no direct equivalent: not present in the 1.0 default_settings() registration group.
# ops["spatial_hp_reg"] = 42

# Gaussian smoothing sigma applied before the high-pass filter in 1P mode.
# 0 = no pre-smoothing.  Default: 0.
# [1.0] ops["pre_smooth"] has no direct equivalent: not present in the 1.0 default_settings() registration group.
# ops["pre_smooth"] = 0

# Use Cellpose segmentation to define ROI masks.
# 0 = disabled (use suite2p's own functional detection, default).
# 1 = segment on max_proj / meanImg
# 2 = segment on meanImg only
# 3 = segment on enhanced meanImg (high-pass filtered)
# 4 = segment on max_proj only
# ⚠️ HIDDEN from main docs page.  Default: 0.
# [1.0] ops["anatomical_only"] has no direct equivalent: Cellpose is now selected through settings["detection"]["algorithm"] (check the accepted value in the 1.0 docs).
# ops["anatomical_only"] = 0

## 4. Check keys

In [ ]:
changed_in_1_0 = [
    # key                              old ops notebook   suite2p 1.0
    ("registration.nimg_init",         "300",             "400"),
    ("registration.batch_size",        "500",             "100"),
    ("detection.algorithm",            "sparse_mode=True","\"sparsery\" (\"sourcery\" for the old False)"),
    ("detection.nbins",                "nbinned",         "renamed"),
    ("detection.highpass_time",        "high_pass",       "renamed"),
    ("extraction.neuropil_coefficient","neucoeff",        "renamed"),
]
print(f"{'Key':<34} {'Old':<20} {'1.0'}")
print("-" * 80)
for name, old, new in changed_in_1_0:
    print(f"{name:<34} {old:<20} {new}")

In [ ]:
# Check that nothing in db / settings is a typo or a removed (pre-1.0) key
bad_s  = find_unknown_keys(settings, default_settings())
bad_db = find_unknown_keys(db, default_db())
if bad_s:  print(f"⚠️  Unknown settings keys: {bad_s}")
if bad_db: print(f"⚠️  Unknown db keys: {bad_db}")
if not bad_s and not bad_db:
    print("✓ All keys in db and settings exist in the suite2p defaults.")

## 5. Run the pipeline

In [ ]:
# db = the dataset, settings = how to process it.
# To process several recordings with the same settings, keep `settings` fixed
# and build one db per recording (see the batch notebook).
run_s2p(db=db, settings=settings)

save_dir = Path(db["save_path0"]) / db["save_folder"]
print("\n✓ suite2p finished!")
print(f"  Results: {save_dir}")

## 6. Load and inspect results

In [ ]:
plane_idx = 0
save_dir  = Path(db["save_path0"]) / db["save_folder"]
plane_dir = save_dir / f"plane{plane_idx}"

settings_out = np.load(plane_dir / "settings.npy",       allow_pickle=True).item()
reg_outputs  = np.load(plane_dir / "reg_outputs.npy",    allow_pickle=True).item()   # meanImg, meanImgE, xoff, yoff, ...
det_outputs  = np.load(plane_dir / "detect_outputs.npy", allow_pickle=True).item()   # max_proj, Vcorr, ...
F       = np.load(plane_dir / "F.npy",      allow_pickle=True)
Fneu    = np.load(plane_dir / "Fneu.npy",   allow_pickle=True)
spks    = np.load(plane_dir / "spks.npy",   allow_pickle=True)
stat    = np.load(plane_dir / "stat.npy",   allow_pickle=True)
iscell  = np.load(plane_dir / "iscell.npy", allow_pickle=True)

fs = settings_out["fs"]
cell_idx = np.where(iscell[:, 0].astype(bool))[0]

print(f"Plane      : {plane_idx}")
print(f"Total ROIs : {len(stat)}")
print(f"Cells      : {len(cell_idx)}")
print(f"Frames     : {F.shape[1]}")
print(f"Frame rate : {fs:.2f} Hz")
print(f"Duration   : {F.shape[1] / fs:.1f} s")

## 7. Quality-control plots

In [ ]:
clip = lambda img: dict(vmin=np.percentile(img, 1), vmax=np.percentile(img, 99))

# ── Mean & enhanced mean images ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(reg_outputs["meanImg"], cmap="gray", **clip(reg_outputs["meanImg"]))
axes[0].set_title("Mean image"); axes[0].axis("off")
if "meanImgE" in reg_outputs:
    axes[1].imshow(reg_outputs["meanImgE"], cmap="gray", **clip(reg_outputs["meanImgE"]))
    axes[1].set_title("Enhanced mean"); axes[1].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# ── Registration shifts ───────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
t = np.arange(len(reg_outputs["yoff"])) / fs
axes[0].plot(t, reg_outputs["yoff"], lw=0.5); axes[0].set_ylabel("Y shift (px)")
axes[0].set_title("Rigid registration shifts")
axes[1].plot(t, reg_outputs["xoff"], lw=0.5, color="C1"); axes[1].set_ylabel("X shift (px)")
axes[1].set_xlabel("Time (s)")
plt.tight_layout(); plt.show()

In [ ]:
# ── Cell centroids on mean image ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(reg_outputs["meanImg"], cmap="gray", **clip(reg_outputs["meanImg"]))
for s in stat[cell_idx]:
    ax.scatter(s["med"][1], s["med"][0], s=5, c="cyan", linewidths=0)
ax.set_title(f"{len(cell_idx)} cells"); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# ── ΔF/F traces ───────────────────────────────────────────────────────────────
N_SHOW = min(20, len(cell_idx))
show_idx = np.random.default_rng(0).choice(cell_idx, size=N_SHOW, replace=False)
F_corr = F[show_idx] - settings_out["extraction"]["neuropil_coefficient"] * Fneu[show_idx]
F0     = np.percentile(F_corr, 8, axis=1, keepdims=True)
dFF    = (F_corr - F0) / (F0 + 1)
t = np.arange(F.shape[1]) / fs
fig, ax = plt.subplots(figsize=(16, N_SHOW * 0.5 + 1))
for i, trace in enumerate(dFF):
    ax.plot(t, trace + i * 3, lw=0.6, color="k")
ax.set_xlabel("Time (s)"); ax.set_ylabel("Cell (ΔF/F)"); ax.set_yticks([])
ax.set_title(f"ΔF/F traces ({N_SHOW} cells)")
plt.tight_layout(); plt.show()

In [ ]:
# ── Classifier score distribution ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(iscell[:, 1], bins=50, edgecolor="k", linewidth=0.3)
ax.axvline(0.5, color="red", linestyle="--", label="default threshold = 0.5")
ax.set_xlabel("Classifier probability (cell)"); ax.set_ylabel("Count")
ax.set_title("ROI classifier scores"); ax.legend()
plt.tight_layout(); plt.show()

## 8. Save the settings (and db) for reproducibility

In [ ]:
# 1) Reusable settings preset: processing parameters only, no paths.
#    Load it in the batch / mesoscope notebooks.
preset_path = Path(SAVE_PATH) / f"{SETTINGS_NAME}.json"
with open(preset_path, "w") as f:
    json.dump(to_jsonable(settings), f, indent=2)
print(f"Settings preset saved to: {preset_path}")

# 2) Dataset description actually used (paths, planes, file list, ...).
db_path = save_dir / "db_used.json"
with open(db_path, "w") as f:
    json.dump(to_jsonable(db), f, indent=2)
print(f"db saved to: {db_path}")

# (suite2p itself also wrote settings.npy and db.npy into the results folder.)

---
## Appendix: old `ops` key → 1.0 location

| old `ops` key | 1.0 |
|---|---|
| `data_path` | `db["data_path"]` |
| `save_path0` | `db["save_path0"]` |
| `fast_disk` | `db["fast_disk"]` |
| `save_folder` | `db["save_folder"]` |
| `look_one_level_down` | `db["look_one_level_down"]` |
| `subfolders` | `db["subfolders"]` |
| `force_sktiff` | `db["force_sktiff"]` |
| `bruker_bidirectional` | `db["bruker_bidirectional"]` |
| `h5py_key` | `db["h5py_key"]` |
| `nwb_driver` | `db["nwb_driver"]` |
| `nwb_series` | `db["nwb_series"]` |
| `nplanes` | `db["nplanes"]` |
| `nchannels` | `db["nchannels"]` |
| `functional_chan` | `db["functional_chan"]` |
| `ignore_flyback` | `db["ignore_flyback"]` |
| `keep_movie_raw` | `db["keep_movie_raw"]` |
| `delete_bin` | `settings["io"]["delete_bin"]` |
| `move_bin` | `settings["io"]["move_bin"]` |
| `save_mat` | `settings["io"]["save_mat"]` |
| `save_NWB` | `settings["io"]["save_NWB"]` |
| `combined` | `settings["io"]["combined"]` |
| `fs` | `settings["fs"]` |
| `tau` | `settings["tau"]` |
| `diameter` | `settings["diameter"]` |
| `do_registration` | `settings["run"]["do_registration"]` |
| `roidetect` | `settings["run"]["do_detection"]` |
| `spikedetect` | `settings["run"]["do_deconvolution"]` |
| `multiplane_parallel` | `settings["run"]["multiplane_parallel"]` |
| `do_bidiphase` | `settings["registration"]["do_bidiphase"]` |
| `bidiphase` | `settings["registration"]["bidiphase"]` |
| `nimg_init` | `settings["registration"]["nimg_init"]` |
| `batch_size` | `settings["registration"]["batch_size"]` |
| `maxregshift` | `settings["registration"]["maxregshift"]` |
| `smooth_sigma` | `settings["registration"]["smooth_sigma"]` |
| `smooth_sigma_time` | `settings["registration"]["smooth_sigma_time"]` |
| `norm_frames` | `settings["registration"]["norm_frames"]` |
| `th_badframes` | `settings["registration"]["th_badframes"]` |
| `subpixel` | `settings["registration"]["subpixel"]` |
| `two_step_registration` | `settings["registration"]["two_step_registration"]` |
| `nonrigid` | `settings["registration"]["nonrigid"]` |
| `block_size` | `settings["registration"]["block_size"]` |
| `snr_thresh` | `settings["registration"]["snr_thresh"]` |
| `maxregshiftNR` | `settings["registration"]["maxregshiftNR"]` |
| `spatial_taper` | `settings["registration"]["spatial_taper"]` |
| `reg_tif` | `settings["registration"]["reg_tif"]` |
| `reg_tif_chan2` | `settings["registration"]["reg_tif_chan2"]` |
| `spatial_scale` | `settings["detection"]["sparsery_settings"]["spatial_scale"]` |
| `spatial_hp_detect` | `settings["detection"]["sparsery_settings"]["highpass_neuropil"]` |
| `connected` | `settings["detection"]["sourcery_settings"]["connected"]` |
| `max_iterations` | `settings["detection"]["sourcery_settings"]["max_iterations"]` |
| `nbinned` | `settings["detection"]["nbins"]` |
| `threshold_scaling` | `settings["detection"]["threshold_scaling"]` |
| `high_pass` | `settings["detection"]["highpass_time"]` |
| `max_overlap` | `settings["detection"]["max_overlap"]` |
| `denoise` | `settings["detection"]["denoise"]` |
| `soma_crop` | `settings["detection"]["soma_crop"]` |
| `chan2_thres` | `settings["detection"]["chan2_threshold"]` |
| `cellprob_threshold` | `settings["detection"]["cellpose_settings"]["cellprob_threshold"]` |
| `flow_threshold` | `settings["detection"]["cellpose_settings"]["flow_threshold"]` |
| `spatial_hp_cp` | `settings["detection"]["cellpose_settings"]["highpass_spatial"]` |
| `pretrained_model` | `settings["detection"]["cellpose_settings"]["cellpose_model"]` |
| `preclassify` | `settings["classification"]["preclassify"]` |
| `use_builtin_classifier` | `settings["classification"]["use_builtin_classifier"]` |
| `classifier_path` | `settings["classification"]["classifier_path"]` |
| `neuropil_extract` | `settings["extraction"]["neuropil_extract"]` |
| `inner_neuropil_radius` | `settings["extraction"]["inner_neuropil_radius"]` |
| `min_neuropil_pixels` | `settings["extraction"]["min_neuropil_pixels"]` |
| `lam_percentile` | `settings["extraction"]["lam_percentile"]` |
| `allow_overlap` | `settings["extraction"]["allow_overlap"]` |
| `neucoeff` | `settings["extraction"]["neuropil_coefficient"]` |
| `baseline` | `settings["dcnv_preprocess"]["baseline"]` |
| `win_baseline` | `settings["dcnv_preprocess"]["win_baseline"]` |
| `sig_baseline` | `settings["dcnv_preprocess"]["sig_baseline"]` |
| `prctile_baseline` | `settings["dcnv_preprocess"]["prctile_baseline"]` |
| `sparse_mode` | `settings["detection"]["algorithm"]` (`"sparsery"` / `"sourcery"`) |
| `align_by_chan` | `settings["registration"]["align_by_chan2"]` (bool) |

**Keys without a direct 1.0 counterpart** (kept as commented-out lines above):

| old `ops` key | note |
|---|---|
| `bruker` | Bruker input is now chosen with db["input_format"] (check the accepted value for Bruker in the 1.0 docs). |
| `mesoscan` | there is no mesoscan flag in db; multi-ROI data is described by db["nrois"], db["lines"], db["dx"], db["dy"] (see the mesoscope notebook). |
| `h5py` | HDF5 input is now chosen with db["input_format"] (check the accepted value for HDF5 in the 1.0 docs) plus db["h5py_key"]. |
| `nwb_file` | NWB input is now chosen with db["input_format"] (check the accepted value for NWB in the 1.0 docs). |
| `frames_include` | not present in the 1.0 default_db() / default_settings(). |
| `bidi_corrected` | not present in the 1.0 default_db() / default_settings(). |
| `aspect` | GUI-only display key; not present in the 1.0 defaults. |
| `force_refImg` | not present in the 1.0 default_settings(). |
| `pad_fft` | not present in the 1.0 default_settings(). |
| `1Preg` | not present in the 1.0 default_settings() registration group. |
| `spatial_hp_reg` | not present in the 1.0 default_settings() registration group. |
| `pre_smooth` | not present in the 1.0 default_settings() registration group. |
| `anatomical_only` | Cellpose is now selected through settings["detection"]["algorithm"] (check the accepted value in the 1.0 docs). |

Sources: the 1.0 `default_settings()` / `default_db()` in `suite2p/parameters.py` and the docs at https://suite2p.readthedocs.io/en/latest/parameters/. Keys not listed in either were treated as removed.